# SingBERT Fine-Tune — NS Sentiment (Stage 5a)

Fine-tunes `zanelim/singbert-large-sg` (BERT-large pre-trained on r/singapore + HardwareZone)
on 8,095 labelled NS Reddit chunks for 3-class sentiment classification.

## Data
- **Train:** `singbert_train.csv` — 8,095 rows (197 human × weight 3.0 + 7,898 LLM × weight 1.0)
- **Eval:** `holdout_test.csv` — 195 evaluable rows (sealed, never seen during training)

## Label distribution (train)
- neutral: 60.0% (4,861)
- negative: 27.5% (2,229)
- positive: 12.4% (1,005)  ← class-weighted to compensate

## Approach
1. Replicate human rows 3× (from `weight` column) before train/val split
2. Compute balanced class weights — boosts positive (~2.7×) and negative (~1.2×)
3. Custom `WeightedTrainer` applies class weights in cross-entropy loss
4. 90/10 train/val split; early stopping on val loss (patience=2)
5. Final eval on sealed `holdout_test.csv` — report accuracy, κ, per-class F1

## Kaggle dataset required
Upload `singbert_train.csv` + `holdout_test.csv` as dataset `ns-sentiment-labels-v1`
before running this notebook.

In [ ]:
# Upgrade transformers + peft together to avoid EncoderDecoderCache import mismatch.
# Kaggle's pre-installed peft requires transformers>=4.43.0; pinning 4.40.2 breaks it.
!pip install -q -U transformers peft datasets scikit-learn

In [ ]:
import os
import glob
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, cohen_kappa_score,
    classification_report, f1_score
)
from sklearn.utils.class_weight import compute_class_weight
from torch.utils.data import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
)

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Auto-discover input CSVs (works regardless of dataset slug name) ───────
def find_input_file(filename: str) -> Path:
    matches = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
    if not matches:
        # Also check working dir (useful for local testing)
        local = Path(filename)
        if local.exists():
            return local
        raise FileNotFoundError(
            f"{filename} not found under /kaggle/input/\n"
            f"Make sure the dataset containing {filename} is attached via Add Data."
        )
    return Path(matches[0])

TRAIN_PATH   = find_input_file("singbert_train.csv")
HOLDOUT_PATH = find_input_file("holdout_test.csv")
print(f"Train  : {TRAIN_PATH}")
print(f"Holdout: {HOLDOUT_PATH}")

OUTPUT_DIR = Path("/kaggle/working/singbert_ns_sentiment")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Model config ───────────────────────────────────────────────────────────
MODEL_NAME = "zanelim/singbert-large-sg"
MAX_LEN    = 256
BATCH_SIZE = 16      # BERT-large fits 16 on T4 16GB
GRAD_ACCUM = 2       # effective batch = 32
LR         = 2e-5
EPOCHS     = 5
VAL_FRAC   = 0.10

# ── Label encoding ─────────────────────────────────────────────────────────
LABEL2ID = {"negative": 0, "neutral": 1, "positive": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\nDevice : {DEVICE}")
print(f"PyTorch: {torch.__version__}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ── Load & prepare training data ───────────────────────────────────────────
raw = pd.read_csv(TRAIN_PATH)
print(f"Raw train rows : {len(raw)}")
print(raw['label'].value_counts())

# Drop any rows with missing text or label
raw = raw.dropna(subset=["text", "label"]).reset_index(drop=True)
raw = raw[raw["label"].isin(LABEL2ID)].reset_index(drop=True)
raw["label_id"] = raw["label"].map(LABEL2ID)

# ── IMPORTANT: split BEFORE replication to prevent val contamination ────────
# Replicating first then splitting puts duplicate rows in both train and val,
# artificially inflating val metrics while holdout remains clean.
train_raw, val_raw = train_test_split(
    raw,
    test_size=VAL_FRAC,
    random_state=SEED,
    stratify=raw["label_id"],
)

# Replicate human rows (weight=3.0) in the TRAIN split only
human_train = train_raw[train_raw["weight"] >= 3.0]
llm_train   = train_raw[train_raw["weight"] <  3.0]
train_df = pd.concat([
    llm_train,
    human_train,   # original
    human_train,   # copy 1
    human_train,   # copy 2
], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)

train_texts  = train_df["text"].tolist()
train_labels = train_df["label_id"].tolist()
val_texts    = val_raw["text"].tolist()
val_labels   = val_raw["label_id"].tolist()

print(f"\nAfter split-then-replicate:")
print(f"  Train : {len(train_texts)}  (human rows ×3 applied here only)")
print(f"  Val   : {len(val_texts)}   (clean — no duplicates of train)")
print(f"\nTrain label dist:")
print(pd.Series(train_labels).map(ID2LABEL).value_counts())
print(f"\nVal label dist:")
print(pd.Series(val_labels).map(ID2LABEL).value_counts())

In [ ]:
# label encoding + train/val split now done in the cell above — nothing to do here
print(f"Train : {len(train_texts)} rows")
print(f"Val   : {len(val_texts)} rows")

In [ ]:
# ── Class weights (on TRAINING split only — not val) ───────────────────────
classes = np.array([0, 1, 2])
cw = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.array(train_labels)
)
# Optionally boost positive a bit more — feel free to tune
POSITIVE_BOOST = 1.0   # set to >1.0 if positive recall still too low after training
cw[2] *= POSITIVE_BOOST

CLASS_WEIGHTS = torch.tensor(cw, dtype=torch.float)
print("Class weights (negative / neutral / positive):")
for cls_id, name in ID2LABEL.items():
    print(f"  {name:<10} {CLASS_WEIGHTS[cls_id]:.4f}")

In [ ]:
# ── Tokenizer ──────────────────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Tokenising train ({len(train_texts)} examples) …")
train_enc = tokenizer(
    train_texts, truncation=True, max_length=MAX_LEN, padding=False
)
print(f"Tokenising val   ({len(val_texts)} examples) …")
val_enc = tokenizer(
    val_texts,   truncation=True, max_length=MAX_LEN, padding=False
)
print("Done.")

In [ ]:
# ── PyTorch Dataset ────────────────────────────────────────────────────────
class SentimentDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels    = labels

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k, v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)


train_dataset = SentimentDataset(train_enc, train_labels)
val_dataset   = SentimentDataset(val_enc,   val_labels)
print(f"Train dataset : {len(train_dataset)}")
print(f"Val dataset   : {len(val_dataset)}")

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
)
model.to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
trainable    = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params    : {total_params:,}")
print(f"Trainable params: {trainable:,}")

In [ ]:
# ── Custom Trainer with class-weighted loss ────────────────────────────────
class WeightedTrainer(Trainer):
    """Applies class weights to cross-entropy loss to handle class imbalance."""

    def __init__(self, class_weights: torch.Tensor, **kwargs):
        super().__init__(**kwargs)
        self._class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels  = inputs.pop("labels")
        outputs = model(**inputs)
        logits  = outputs.logits

        loss_fn = nn.CrossEntropyLoss(
            weight=self._class_weights.to(logits.device)
        )
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss


# ── Evaluation metrics ─────────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average="weighted", zero_division=0)
    kappa = cohen_kappa_score(labels, preds)
    return {"accuracy": acc, "weighted_f1": f1, "kappa": kappa}


print("WeightedTrainer defined.")

In [ ]:
# ── Training arguments ─────────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir                  = str(OUTPUT_DIR),
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE * 2,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LR,
    weight_decay                = 0.01,
    warmup_ratio                = 0.06,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "kappa",
    greater_is_better           = True,
    fp16                        = torch.cuda.is_available(),
    dataloader_num_workers      = 2,
    logging_steps               = 50,
    report_to                   = "none",
    seed                        = SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# transformers>=4.46 renamed `tokenizer` → `processing_class` in Trainer
import transformers
_trainer_kwargs = dict(
    class_weights   = CLASS_WEIGHTS,
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    data_collator   = data_collator,
    compute_metrics = compute_metrics,
    callbacks       = [EarlyStoppingCallback(early_stopping_patience=2)],
)
tv = tuple(int(x) for x in transformers.__version__.split(".")[:2])
if tv >= (4, 46):
    _trainer_kwargs["processing_class"] = tokenizer
else:
    _trainer_kwargs["tokenizer"] = tokenizer

trainer = WeightedTrainer(**_trainer_kwargs)

print("Trainer configured.")
steps_per_epoch = len(train_dataset) // (BATCH_SIZE * GRAD_ACCUM)
print(f"~{steps_per_epoch} optimizer steps/epoch × {EPOCHS} epochs max")

In [ ]:
# ── Train ──────────────────────────────────────────────────────────────────
print("Starting training …")
train_result = trainer.train()
print("\nTraining complete.")
print(f"  Total steps     : {train_result.global_step}")
print(f"  Training loss   : {train_result.training_loss:.4f}")

# Log val metrics at best checkpoint
metrics = trainer.evaluate()
print(f"\nBest val metrics:")
for k, v in metrics.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

In [ ]:
# ── Holdout evaluation — sealed test set ───────────────────────────────────
print("Loading sealed holdout set …")
holdout_df = pd.read_csv(HOLDOUT_PATH)
holdout_df = holdout_df[holdout_df["human_label"].isin(LABEL2ID)].reset_index(drop=True)
print(f"Evaluable holdout rows: {len(holdout_df)}")
print(holdout_df["human_label"].value_counts())

# Tokenise
holdout_enc = tokenizer(
    holdout_df["text"].tolist(),
    truncation=True,
    max_length=MAX_LEN,
    padding=False,
)
holdout_labels = holdout_df["human_label"].map(LABEL2ID).tolist()
holdout_dataset = SentimentDataset(holdout_enc, holdout_labels)

# Inference
holdout_preds_raw = trainer.predict(holdout_dataset)
holdout_preds = np.argmax(holdout_preds_raw.predictions, axis=-1)
holdout_true  = holdout_labels

# Decode
preds_named = [ID2LABEL[p] for p in holdout_preds]
true_named  = holdout_df["human_label"].tolist()

# Scores
acc   = accuracy_score(holdout_true, holdout_preds)
kappa = cohen_kappa_score(holdout_true, holdout_preds)

print()
print("═" * 62)
print("  Holdout Evaluation — SingBERT fine-tuned")
print("═" * 62)
print(f"  Rows evaluated : {len(holdout_true)}")
print(f"  Accuracy       : {acc:.3f}  ({acc*100:.1f}%)")
print(f"  Cohen's Kappa  : {kappa:.3f}")
print()
print(classification_report(
    true_named, preds_named,
    labels=["negative", "neutral", "positive"],
    digits=3
))
print("═" * 62)

In [ ]:
# ── Save holdout predictions ───────────────────────────────────────────────
holdout_df["singbert_label"] = preds_named
holdout_df["correct"]        = holdout_df["human_label"] == holdout_df["singbert_label"]

out_path = OUTPUT_DIR / "holdout_eval_singbert.csv"
holdout_df.to_csv(out_path, index=False)
print(f"Saved → {out_path}")

# Also save a JSON summary
summary = {
    "model"    : MODEL_NAME,
    "rows"     : len(holdout_true),
    "accuracy" : round(acc,   4),
    "kappa"    : round(kappa, 4),
    "max_len"  : MAX_LEN,
    "lr"       : LR,
    "epochs_trained": train_result.global_step // steps_per_epoch,
    "positive_boost": POSITIVE_BOOST,
}
with open(OUTPUT_DIR / "eval_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

In [ ]:
# ── Save model + tokenizer ─────────────────────────────────────────────────
# Best checkpoint already loaded by load_best_model_at_end=True
model_out = OUTPUT_DIR / "best_model"
trainer.save_model(str(model_out))
tokenizer.save_pretrained(str(model_out))
print(f"Model saved → {model_out}")
print("\nContents:")
for f in sorted(model_out.iterdir()):
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:<40} {size_mb:>8.1f} MB")

In [ ]:
# ── Zip for Kaggle dataset upload ──────────────────────────────────────────
import shutil
zip_path = str(OUTPUT_DIR.parent / "singbert_ns_sentiment")
shutil.make_archive(zip_path, "zip", str(model_out))
print(f"Zipped → {zip_path}.zip")
zip_size = Path(zip_path + ".zip").stat().st_size / 1e6
print(f"Size   : {zip_size:.0f} MB")

print()
print("Next step: upload singbert_ns_sentiment.zip to a new Kaggle dataset")
print("Then build kaggle_infer_singbert_v1.ipynb for full 737k inference.")

## Tuning guide

If positive F1 is still below 60% after training:
1. Set `POSITIVE_BOOST = 1.5` (cell 5) and retrain
2. Or lower the positive class decision threshold (see cell below)

If accuracy < 80%:
- Try `LR = 1e-5` (slower but can help BERT-large fine-tuning)
- Try `EPOCHS = 8` with patience=3

Good expected ranges based on LLM baseline (81.0% acc, κ=0.653):
- **SingBERT target: 83–87% accuracy, κ > 0.70**

In [ ]:
# ── Optional: threshold sweep for positive class ───────────────────────────
# Run this cell ONLY if positive recall is too low.
# Lowers the decision boundary for positive to increase recall at cost of precision.

import torch.nn.functional as F

probs = F.softmax(
    torch.tensor(holdout_preds_raw.predictions, dtype=torch.float32), dim=-1
).numpy()

print("Threshold sweep for positive class (class index 2):")
print(f"{'Threshold':>10}  {'Precision':>10}  {'Recall':>10}  {'F1':>8}  {'Acc':>8}")

from sklearn.metrics import precision_recall_fscore_support

for thr in np.arange(0.25, 0.55, 0.05):
    # Predict positive if prob_positive >= thr, otherwise argmax of neg/neu
    preds_thr = []
    for row in probs:
        if row[2] >= thr:
            preds_thr.append(2)
        else:
            preds_thr.append(np.argmax(row[:2]))  # neg or neu
    p, r, f, _ = precision_recall_fscore_support(
        holdout_true, preds_thr,
        labels=[2], average=None, zero_division=0
    )
    acc_t = accuracy_score(holdout_true, preds_thr)
    print(f"  {thr:.2f}        {p[0]:.3f}       {r[0]:.3f}      {f[0]:.3f}   {acc_t:.3f}")